In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
from scipy.ndimage import distance_transform_edt
import random
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
from rasterio.transform import Affine
import xgboost as xgb
from xgboost import XGBClassifier
import os, re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import LeaveOneGroupOut

from scipy.spatial.distance import pdist, squareform
import joblib
from pathlib import Path

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, cohen_kappa_score


In [2]:
names = locals()

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
FEATURE_DIR = RESULTS / "features"

In [ ]:
region_mask = rio.open(RESULTS / 'region-masks' / 'region-mask-1km.tif').read(1)

In [ ]:
feature_list_1km = dict({
    'elevation': FEATURE_DIR / 'elevation-1km.tif',
    'slope':     FEATURE_DIR / 'slope-1km.tif', 
    'population': RESULTS / 'population-raster' / 'population-2020-1km.tif',
    'accessibility': FEATURE_DIR / 'accessibility-1km.tif',
    'D2water': FEATURE_DIR / 'distance-to-water-1km.tif', 
    'forest': FEATURE_DIR / 'forest-2020-1km.tif',
    'cropland': FEATURE_DIR / 'cropland-2019-1km.tif', 
    'water': FEATURE_DIR / 'water-mask-1km.tif', 
    'su': FEATURE_DIR / 'su-1km.tif',
    'vb': FEATURE_DIR / 'vb-1km.tif',
    'ss': FEATURE_DIR / 'ss-1km.tif',
    'pb': FEATURE_DIR / 'pb-1km.tif',
    'sm': FEATURE_DIR / 'sm-1km.tif', 
    'sc': FEATURE_DIR / 'sc-1km.tif', 
    'va': FEATURE_DIR / 'va-1km.tif',
    'mt': FEATURE_DIR / 'mt-1km.tif',
    'pa': FEATURE_DIR / 'pa-1km.tif',
    'vi': FEATURE_DIR / 'vi-1km.tif',
    'wb': FEATURE_DIR / 'wb-1km.tif',
    'py': FEATURE_DIR / 'py-1km.tif',
    'pi': FEATURE_DIR / 'pi-1km.tif',
    'ev': FEATURE_DIR / 'ev-1km.tif',
    'nd': FEATURE_DIR / 'nd-1km.tif', 
    'ig': FEATURE_DIR / 'ig-1km.tif', 
    'PA_Ia': RESULTS / 'protected-area-rasters' / 'protected-area-Ia-mask.tif',
    'PA_Ib': RESULTS / 'protected-area-rasters' / 'protected-area-Ib-mask.tif',
    'PA_II': RESULTS / 'protected-area-rasters' / 'protected-area-II-mask.tif',
    'PA_III': RESULTS / 'protected-area-rasters' / 'protected-area-III-mask.tif',
    'PA_IV': RESULTS / 'protected-area-rasters' / 'protected-area-IV-mask.tif',
    'PA_V': RESULTS / 'protected-area-rasters' / 'protected-area-V-mask.tif',
    'PA_VI': RESULTS / 'protected-area-rasters' / 'protected-area-VI-mask.tif',
    'urban': FEATURE_DIR / 'urban-area-2020-1km.tif',
    'bare': FEATURE_DIR / 'bare-land-2020-1km.tif',
    'building_stock': FEATURE_DIR / 'building-mineral-stock-1km.tif',
})

In [ ]:
def compile_samples_for_region_v2(
    the_region,
    selected_features,
    res='1km',
    buffer_min_km=30,          # was 50 — narrower min lets us include more candidates
    buffer_max_km=300,         # was 100 — wider ring captures broader environmental gradient
    far_fraction=0.7,          # 70% from buffer ring, 30% from country background
    pa_ratio=3,                # n_pseudo per country = 3 × n_presence (no 1:1 cap)
    min_presences_per_country=5,
    seed=42,
    verbose=False,
):
    """
    Two-tier pseudo-absence sampling for a given region.
      Tier 1 (far_fraction): buffer ring [buffer_min_km, buffer_max_km] from any presence.
      Tier 2 (1 - far_fraction): country background outside the min buffer.
      Tier 1 shortfall is automatically topped up from Tier 2.
    Returns: X, Y, country_ids, region_ids, bbox, region_mask, value_mask, log_df
    """
    rng = np.random.default_rng(seed + the_region)

    region_mask_  = rio.open(RESULTS / 'region-masks' / 'region-mask-{res}.tif')
    country_rast_ = rio.open(RESULTS / 'country-rasters' / 'state-raster-eck4-{res}.tif')
    global_quarry_area = rio.open(RESULTS / 'quarry-area-rasters' / 'global-aggregate-quarry-area-raster-{res}.tif').read(1)

    region_full   = region_mask_.read(1)
    country_full  = country_rast_.read(1)
    feature_list  = names[f'feature_list_{res}']

    # --- Window the region ---
    rr = np.where(region_full == the_region)
    i_1, i_2 = np.min(rr[0]), np.max(rr[0])
    j_1, j_2 = np.min(rr[1]), np.max(rr[1])
    a_window = ((i_1, i_2), (j_1, j_2))

    presence_raw   = global_quarry_area[i_1:i_2, j_1:j_2]
    region_mask    = (region_full[i_1:i_2, j_1:j_2] == the_region)
    country_mask_w = country_full[i_1:i_2, j_1:j_2]
    presence_in_region = np.where(
        region_mask & (presence_raw > 0), 1, 0
    ).astype(np.uint8)

    # --- Distance transform & buffer in pixels ---
    r = int(re.findall(r'\d+', res)[0])
    D_min, D_max = buffer_min_km // r, buffer_max_km // r
    distance_from_quarry = distance_transform_edt(presence_in_region == 0)

    pseudoabs = np.zeros_like(presence_in_region, dtype=np.uint8)
    log_records = []

    country_ids = np.unique(country_mask_w[region_mask])
    country_ids = country_ids[country_ids > 0]

    for cid in tqdm(country_ids, desc=f'Region {the_region}', disable=not verbose):
        country_local = (country_mask_w == cid) & region_mask  # KEY: restrict to region
        presence_local = (presence_in_region == 1) & country_local
        n_pres = int(presence_local.sum())

        if n_pres < min_presences_per_country:
            continue

        n_target = pa_ratio * n_pres
        n_far_target  = int(round(far_fraction * n_target))
        n_near_target = n_target - n_far_target

        # ---- Tier 1: buffer ring ----
        far_idx = np.where(
            (distance_from_quarry >= D_min) &
            (distance_from_quarry <= D_max) &
            country_local &
            (presence_in_region == 0) &
            (pseudoabs == 0)
        )
        n_far_avail = len(far_idx[0])
        n_far = min(n_far_target, n_far_avail)
        if n_far > 0:
            sel = rng.choice(n_far_avail, size=n_far, replace=False)
            pseudoabs[far_idx[0][sel], far_idx[1][sel]] = 1

        # ---- Tier 2: country background (outside min buffer) ----
        deficit = n_far_target - n_far
        n_near_request = n_near_target + deficit

        bg_idx = np.where(
            country_local &
            (presence_in_region == 0) &
            (pseudoabs == 0) &
            (distance_from_quarry >= D_min)
        )
        n_bg_avail = len(bg_idx[0])
        n_near = min(n_near_request, n_bg_avail)
        if n_near > 0:
            sel = rng.choice(n_bg_avail, size=n_near, replace=False)
            pseudoabs[bg_idx[0][sel], bg_idx[1][sel]] = 1

        log_records.append({
            'region': the_region, 'country': int(cid),
            'n_presence': n_pres,
            'n_pseudo_far': n_far, 'n_pseudo_near': n_near,
            'n_pseudo_total': n_far + n_near,
            'ratio_pseudo_to_presence': (n_far + n_near) / n_pres,
        })

    # --- Build labels & feature matrix ---
    label_mask  = (presence_in_region == 1) | (pseudoabs == 1)
    label_array = np.where(
        presence_in_region == 1, 1,
        np.where(pseudoabs == 1, 0, np.nan)
    ).astype(np.float32)

    feature_values = []
    for feat in selected_features:
        src = rio.open(feature_list[feat])
        gg  = src.read(1, window=a_window)
        feature_values.append(gg[label_mask])

    Y_ = label_array[label_mask]
    X_ = np.array(feature_values).T
    valid = ~np.isnan(X_).any(axis=1)
    X = X_[valid]
    Y = Y_[valid]

    country_ids_arr = country_mask_w[label_mask][valid]
    region_ids_arr  = np.full(len(Y), the_region, dtype=np.int16)

    vm = np.where(label_mask)
    value_mask = (vm[0][valid], vm[1][valid])

    log_df = pd.DataFrame(log_records)
    return X, Y, country_ids_arr, region_ids_arr, (i_1, i_2, j_1, j_2), region_mask, value_mask, log_df

In [8]:
selected_features = ['elevation', 'slope', 'population', 'accessibility', 'D2water', 'forest', 'cropland', 'water', 'su', 'vb', 'ss', 'pb', 'sm', 'sc', 'va', 'mt', 'pa', 'vi', 'wb', 'py', 'pi', 'ev', 'nd', 'ig', 'PA_Ia', 'PA_Ib', 'PA_II', 'PA_III', 'PA_IV', 'PA_V', 'PA_VI', 'urban', 'bare', 'building_stock']
print(len(selected_features)) 

34


In [17]:
X_list, Y_list, country_list, region_list, log_list = [], [], [], [], []

for region_id in tqdm(range(1, 16), desc='Compiling regions'):
    X_r, Y_r, c_r, r_r, *_ , log_r = compile_samples_for_region_v2(
        region_id, selected_features, res='1km', verbose=False
    )
    X_list.append(X_r); Y_list.append(Y_r)
    country_list.append(c_r); region_list.append(r_r)
    log_list.append(log_r)

X_global       = np.vstack(X_list)
Y_global       = np.concatenate(Y_list)
country_global = np.concatenate(country_list)
region_global  = np.concatenate(region_list)

# Unique group ID for GroupKFold (avoids country-ID collisions across regions)
group_global = np.array([f'{r}_{c}' for r, c in zip(region_global, country_global)])

# Add region as one-hot (recommended over integer for sklearn RF)
region_oh = pd.get_dummies(pd.Series(region_global, name='region'), prefix='reg').values
X_global_full = np.hstack([X_global, region_oh])

feature_names_full = list(selected_features) + [f'reg_{i}' for i in sorted(np.unique(region_global))]

print(f'Total samples: {len(Y_global):,}')
print(f'  Presences:        {int(Y_global.sum()):,}')
print(f'  Pseudo-absences:  {int((Y_global == 0).sum()):,}')
print(f'  Pseudo:Presence:  {(Y_global == 0).sum() / Y_global.sum():.2f}')
print(f'  Unique groups (region_country) for CV: {len(np.unique(group_global))}')
print(f'  Feature dim: {X_global_full.shape[1]} ({len(selected_features)} env + {region_oh.shape[1]} region one-hot)')

Compiling regions: 100%|██████████| 15/15 [07:35<00:00, 30.40s/it]


Total samples: 601,390
  Presences:        195,765
  Pseudo-absences:  405,625
  Pseudo:Presence:  2.07
  Unique groups (region_country) for CV: 312
  Feature dim: 49 (34 env + 15 region one-hot)


In [18]:
log_all = pd.concat(log_list, ignore_index=True)

# Sanity-check the dense-quarry country problem from your original
print('Per-region totals:')
print(log_all.groupby('region')[['n_presence','n_pseudo_total']].sum())

print('\nCountries where Tier 1 (buffer ring) was insufficient and Tier 2 topped up:')
shortfall = log_all[log_all['n_pseudo_far'] < 0.7 * log_all['n_pseudo_total'] * (0.7/0.7)]
# i.e. where the actual far share dropped below the 70% target
log_all['far_share'] = log_all['n_pseudo_far'] / log_all['n_pseudo_total']
print(log_all[log_all['far_share'] < 0.5][['region','country','n_presence','n_pseudo_far','n_pseudo_near','far_share']])

print(f'\nMean pseudo:presence ratio across countries: {log_all["ratio_pseudo_to_presence"].mean():.2f}')

Per-region totals:
        n_presence  n_pseudo_total
region                            
1            11602           33870
2            15780           46709
3             4759           14277
4             3741           11103
5             1314            3942
6             1947            5841
7             1575            4641
8            75846           67367
9             2804            8385
10            2896            8599
11            7369           22107
12            1368            4065
13           20416           60178
14           37390           93969
15            7651           22953

Countries where Tier 1 (buffer ring) was insufficient and Tier 2 topped up:
Empty DataFrame
Columns: [region, country, n_presence, n_pseudo_far, n_pseudo_near, far_share]
Index: []

Mean pseudo:presence ratio across countries: 2.41


In [21]:
N_SPLITS = 5
cv = GroupKFold(n_splits=N_SPLITS)

fold_records = []      # one row per (fold, region) — for reporting
oof_pred = np.full(len(Y_global), np.nan)  # out-of-fold predictions

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X_global_full, Y_global, groups=group_global), 1
):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')
    print(f'  Train: {len(train_idx):,} samples, '
          f'{len(np.unique(group_global[train_idx]))} groups')
    print(f'  Test:  {len(test_idx):,} samples, '
          f'{len(np.unique(group_global[test_idx]))} groups')

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        class_weight='balanced',   # handles 1:3 imbalance
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_global_full[train_idx], Y_global[train_idx])

    y_pred_prob = model.predict_proba(X_global_full[test_idx])[:, 1]
    y_pred_bin  = (y_pred_prob >= 0.5).astype(int)
    oof_pred[test_idx] = y_pred_prob

    # Per-region metrics on this fold's test set
    test_regions = region_global[test_idx]
    for r in np.unique(test_regions):
        mask = test_regions == r
        if mask.sum() < 20 or len(np.unique(Y_global[test_idx][mask])) < 2:
            continue  # skip tiny or single-class slices
        y_true_r = Y_global[test_idx][mask]
        y_prob_r = y_pred_prob[mask]
        y_bin_r  = y_pred_bin[mask]
        fold_records.append({
            'fold': fold, 'region': int(r), 'n': int(mask.sum()),
            'auc': roc_auc_score(y_true_r, y_prob_r),
            'ap':  average_precision_score(y_true_r, y_prob_r),
            'f1':  f1_score(y_true_r, y_bin_r),
            'kappa': cohen_kappa_score(y_true_r, y_bin_r),
        })

# Aggregate
results_df = pd.DataFrame(fold_records)
print('\n===== Per-region performance (mean across folds) =====')
print(results_df.groupby('region')[['auc','ap','f1','kappa']].mean().round(3))
print('\n===== Global summary (sample-weighted) =====')
print(results_df[['auc','ap','f1','kappa']].mean().round(3))


=== Fold 1/5 ===
  Train: 481,112 samples, 255 groups
  Test:  120,278 samples, 57 groups

=== Fold 2/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

=== Fold 3/5 ===
  Train: 481,112 samples, 249 groups
  Test:  120,278 samples, 63 groups

=== Fold 4/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

=== Fold 5/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

===== Per-region performance (mean across folds) =====
          auc     ap     f1  kappa
region                            
1       0.843  0.673  0.595  0.490
2       0.824  0.636  0.531  0.421
3       0.796  0.648  0.339  0.267
4       0.752  0.585  0.410  0.307
5       0.684  0.467  0.416  0.264
6       0.765  0.541  0.427  0.299
7       0.858  0.712  0.547  0.444
8       0.878  0.859  0.789  0.548
9       0.862  0.720  0.519  0.421
10      0.861  0.717  0.623  0.478
11      0.844  0.650  0.493  0.399
12      0.817  0.666  0.528  0.411

In [23]:
N_SPLITS = 5
cv = GroupKFold(n_splits=N_SPLITS)

fold_records = []      # one row per (fold, region) — for reporting
oof_pred = np.full(len(Y_global), np.nan)  # out-of-fold predictions

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X_global_full, Y_global, groups=group_global), 1
):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')
    print(f'  Train: {len(train_idx):,} samples, '
          f'{len(np.unique(group_global[train_idx]))} groups')
    print(f'  Test:  {len(test_idx):,} samples, '
          f'{len(np.unique(group_global[test_idx]))} groups')

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=20,
        class_weight='balanced',   # handles 1:3 imbalance
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_global_full[train_idx], Y_global[train_idx])

    y_pred_prob = model.predict_proba(X_global_full[test_idx])[:, 1]
    y_pred_bin  = (y_pred_prob >= 0.5).astype(int)
    oof_pred[test_idx] = y_pred_prob

    # Per-region metrics on this fold's test set
    test_regions = region_global[test_idx]
    for r in np.unique(test_regions):
        mask = test_regions == r
        if mask.sum() < 20 or len(np.unique(Y_global[test_idx][mask])) < 2:
            continue  # skip tiny or single-class slices
        y_true_r = Y_global[test_idx][mask]
        y_prob_r = y_pred_prob[mask]
        y_bin_r  = y_pred_bin[mask]
        fold_records.append({
            'fold': fold, 'region': int(r), 'n': int(mask.sum()),
            'auc': roc_auc_score(y_true_r, y_prob_r),
            'ap':  average_precision_score(y_true_r, y_prob_r),
            'f1':  f1_score(y_true_r, y_bin_r),
            'kappa': cohen_kappa_score(y_true_r, y_bin_r),
        })

# Aggregate
results_df = pd.DataFrame(fold_records)
print('\n===== Per-region performance (mean across folds) =====')
print(results_df.groupby('region')[['auc','ap','f1','kappa']].mean().round(3))
print('\n===== Global summary (sample-weighted) =====')
print(results_df[['auc','ap','f1','kappa']].mean().round(3))


=== Fold 1/5 ===
  Train: 481,112 samples, 255 groups
  Test:  120,278 samples, 57 groups

=== Fold 2/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

=== Fold 3/5 ===
  Train: 481,112 samples, 249 groups
  Test:  120,278 samples, 63 groups

=== Fold 4/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

=== Fold 5/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

===== Per-region performance (mean across folds) =====
          auc     ap     f1  kappa
region                            
1       0.851  0.687  0.639  0.527
2       0.840  0.662  0.591  0.466
3       0.819  0.671  0.410  0.321
4       0.753  0.591  0.496  0.361
5       0.682  0.474  0.483  0.287
6       0.798  0.570  0.519  0.364
7       0.876  0.743  0.652  0.528
8       0.888  0.869  0.788  0.533
9       0.867  0.725  0.597  0.469
10      0.866  0.742  0.639  0.482
11      0.839  0.650  0.486  0.361
12      0.819  0.669  0.540  0.352

In [26]:
N_SPLITS = 5
cv = GroupKFold(n_splits=N_SPLITS)

fold_records = []      # one row per (fold, region) — for reporting
oof_pred = np.full(len(Y_global), np.nan)  # out-of-fold predictions

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X_global_full, Y_global, groups=group_global), 1
):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')
    print(f'  Train: {len(train_idx):,} samples, '
          f'{len(np.unique(group_global[train_idx]))} groups')
    print(f'  Test:  {len(test_idx):,} samples, '
          f'{len(np.unique(group_global[test_idx]))} groups')

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=20,
        min_samples_leaf=5,
        class_weight='balanced',   # handles 1:3 imbalance
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_global_full[train_idx], Y_global[train_idx])

    y_pred_prob = model.predict_proba(X_global_full[test_idx])[:, 1]
    y_pred_bin  = (y_pred_prob >= 0.5).astype(int)
    oof_pred[test_idx] = y_pred_prob

    # Per-region metrics on this fold's test set
    test_regions = region_global[test_idx]
    for r in np.unique(test_regions):
        mask = test_regions == r
        if mask.sum() < 20 or len(np.unique(Y_global[test_idx][mask])) < 2:
            continue  # skip tiny or single-class slices
        y_true_r = Y_global[test_idx][mask]
        y_prob_r = y_pred_prob[mask]
        y_bin_r  = y_pred_bin[mask]
        fold_records.append({
            'fold': fold, 'region': int(r), 'n': int(mask.sum()),
            'auc': roc_auc_score(y_true_r, y_prob_r),
            'ap':  average_precision_score(y_true_r, y_prob_r),
            'f1':  f1_score(y_true_r, y_bin_r),
            'kappa': cohen_kappa_score(y_true_r, y_bin_r),
        })

# Aggregate
results_df = pd.DataFrame(fold_records)
print('\n===== Per-region performance (mean across folds) =====')
print(results_df.groupby('region')[['auc','ap','f1','kappa']].mean().round(3))
print('\n===== Global summary (sample-weighted) =====')
print(results_df[['auc','ap','f1','kappa']].mean().round(3))


=== Fold 1/5 ===
  Train: 481,112 samples, 255 groups
  Test:  120,278 samples, 57 groups

=== Fold 2/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

=== Fold 3/5 ===
  Train: 481,112 samples, 249 groups
  Test:  120,278 samples, 63 groups

=== Fold 4/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

=== Fold 5/5 ===
  Train: 481,112 samples, 248 groups
  Test:  120,278 samples, 64 groups

===== Per-region performance (mean across folds) =====
          auc     ap     f1  kappa
region                            
1       0.852  0.688  0.646  0.527
2       0.842  0.674  0.600  0.469
3       0.813  0.673  0.480  0.383
4       0.760  0.597  0.511  0.366
5       0.699  0.487  0.492  0.282
6       0.800  0.575  0.532  0.374
7       0.877  0.745  0.655  0.525
8       0.889  0.872  0.786  0.522
9       0.867  0.728  0.598  0.461
10      0.864  0.751  0.643  0.483
11      0.836  0.652  0.491  0.349
12      0.826  0.671  0.560  0.352

In [27]:
final_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
final_model.fit(X_global_full, Y_global)

joblib.dump(final_model, RESULTS / 'trained-models' / 'rf-global.pkl')
joblib.dump(feature_names_full, RESULTS / 'trained-models' / 'rf-global-features.pkl')
print('Saved global RF model.')

Saved global RF model.


20260506

In [ ]:
# RF settings — match your final-model choices
RF_KWARGS = dict(
    n_estimators=400,
    max_depth=20,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

def pick_cv(n_groups):
    """Replicates your old logic: GroupKFold if ≥5 groups, else LeaveOneGroupOut."""
    if n_groups >= 5:
        return GroupKFold(n_splits=min(5, n_groups)), 'GroupKFold'
    else:
        return LeaveOneGroupOut(), 'LeaveOneGroupOut'

In [40]:
region_abbr = dict({
    1: 'ME', 
    2: 'LAC',
    3: 'NAF', 
    4: 'WAF',
    5: 'EAF',
    6: 'CAF',
    7: 'SAF',
    8: 'EU', 
    9: 'SEA', 
    10: 'OC',
    11: 'SA', 
    12: 'EA', 
    13: 'CNA', 
    14: 'NA',
    15: 'CHN'
})

In [41]:
results_per_region = []

for region_id in range(1, 16):
    # Mask for samples in this region
    region_mask = (region_global == region_id)
    if region_mask.sum() == 0:
        continue

    # Groups for THIS region only (countries, or states/provinces for NA & China)
    # NB: assumes country_global already encodes states/provinces for regions 14 and 15
    groups_in_region = group_global[region_mask]
    unique_groups = np.unique(groups_in_region)
    n_groups = len(unique_groups)

    cv, cv_name = pick_cv(n_groups)
    print(f'\nRegion {region_id} ({region_abbr[region_id]}): '
          f'{n_groups} groups, using {cv_name}')

    # Indices into the full dataset for samples in this region
    region_sample_idx = np.where(region_mask)[0]
    X_region = X_global_full[region_sample_idx]
    Y_region = Y_global[region_sample_idx]
    groups_region = groups_in_region

    for fold, (_, test_local_idx) in enumerate(
        cv.split(X_region, Y_region, groups=groups_region), 1
    ):
        # Translate local test indices back to global dataset positions
        test_global_idx = region_sample_idx[test_local_idx]
        held_out_groups = np.unique(group_global[test_global_idx])

        # Train on EVERYTHING globally except samples in held-out groups
        train_mask = ~np.isin(group_global, held_out_groups)
        test_mask  = np.isin(group_global, held_out_groups) & region_mask
        # The & region_mask ensures we evaluate ONLY on this region's
        # held-out countries, not other regions' samples in the same group
        # (group strings like "8_60" are unique by construction, so this is
        # belt-and-braces — but safer)

        if test_mask.sum() < 20 or len(np.unique(Y_global[test_mask])) < 2:
            print(f'  Fold {fold}: skipped (too few samples or single class)')
            continue

        model = RandomForestClassifier(**RF_KWARGS)
        model.fit(X_global_full[train_mask], Y_global[train_mask])

        prob = model.predict_proba(X_global_full[test_mask])[:, 1]
        pred = (prob >= 0.5).astype(int)
        y_true = Y_global[test_mask]

        results_per_region.append({
            'region': region_id,
            'fold': fold,
            'n_train': int(train_mask.sum()),
            'n_test':  int(test_mask.sum()),
            'n_test_groups': len(held_out_groups),
            'auc':   roc_auc_score(y_true, prob),
            'ap':    average_precision_score(y_true, prob),
            'f1':    f1_score(y_true, pred),
            'kappa': cohen_kappa_score(y_true, pred),
        })
        print(f'  Fold {fold}: train={train_mask.sum():,}, test={test_mask.sum():,} '
              f'({len(held_out_groups)} held-out groups), '
              f'AUC={results_per_region[-1]["auc"]:.3f}, '
              f'κ={results_per_region[-1]["kappa"]:.3f}')

results_per_region_df = pd.DataFrame(results_per_region)


Region 1 (ME): 16 groups, using GroupKFold
  Fold 1: train=584,874, test=16,516 (1 held-out groups), AUC=0.815, κ=0.391
  Fold 2: train=592,976, test=8,414 (1 held-out groups), AUC=0.780, κ=0.458
  Fold 3: train=593,936, test=7,454 (1 held-out groups), AUC=0.770, κ=0.319
  Fold 4: train=594,893, test=6,497 (6 held-out groups), AUC=0.942, κ=0.704
  Fold 5: train=594,928, test=6,462 (7 held-out groups), AUC=0.875, κ=0.561

Region 2 (LAC): 46 groups, using GroupKFold
  Fold 1: train=581,191, test=20,199 (1 held-out groups), AUC=0.858, κ=0.525
  Fold 2: train=590,330, test=11,060 (1 held-out groups), AUC=0.822, κ=0.388
  Fold 3: train=591,074, test=10,316 (15 held-out groups), AUC=0.878, κ=0.546
  Fold 4: train=591,074, test=10,316 (16 held-out groups), AUC=0.875, κ=0.536
  Fold 5: train=591,075, test=10,315 (13 held-out groups), AUC=0.818, κ=0.282

Region 3 (NAF): 7 groups, using GroupKFold
  Fold 1: train=592,592, test=8,798 (1 held-out groups), AUC=0.791, κ=0.505
  Fold 2: train=597,50

In [42]:
# Mean ± SD per region
agg = (
    results_per_region_df
    .groupby('region')[['auc','ap','f1','kappa']]
    .agg(['mean','std'])
    .round(3)
)
print('===== Per-region performance: mean ± SD across folds =====')
print(agg)

# Manuscript-ready format
formatted = pd.DataFrame(index=agg.index)
formatted['region_name'] = [region_abbr[r] for r in agg.index]
for metric in ['auc','ap','f1','kappa']:
    formatted[metric] = (
        agg[(metric,'mean')].round(3).astype(str) + ' ± '
        + agg[(metric,'std')].round(3).astype(str)
    )
print('\n===== Manuscript-ready format =====')
print(formatted)

# Overall
print('\n===== Global summary across all folds =====')
for m in ['auc','ap','f1','kappa']:
    print(f'  {m}: {results_per_region_df[m].mean():.3f} '
          f'± {results_per_region_df[m].std():.3f}')

# Save
results_per_region_df.to_csv('../results/cv_results_per_region_global_model.csv', index=False)

===== Per-region performance: mean ± SD across folds =====
          auc            ap            f1         kappa       
         mean    std   mean    std   mean    std   mean    std
region                                                        
1       0.836  0.072  0.663  0.122  0.617  0.133  0.487  0.151
2       0.850  0.029  0.682  0.050  0.572  0.132  0.455  0.116
3       0.800  0.040  0.642  0.083  0.544  0.114  0.430  0.112
4       0.794  0.114  0.628  0.156  0.554  0.092  0.412  0.130
5       0.748  0.198  0.551  0.209  0.500  0.217  0.347  0.199
6       0.800  0.095  0.596  0.139  0.563  0.152  0.410  0.196
7       0.859  0.049  0.714  0.097  0.650  0.092  0.506  0.137
8       0.874  0.062  0.847  0.130  0.774  0.164  0.495  0.224
9       0.834  0.066  0.672  0.093  0.601  0.082  0.447  0.106
10      0.825  0.112  0.726  0.150  0.643  0.143  0.450  0.233
11      0.866  0.032  0.707  0.090  0.543  0.138  0.418  0.150
12      0.824  0.054  0.669  0.036  0.549  0.077  0.327  0.

In [ ]:
# Save plotting data for Supplementary Figure 12 (drawn in 9-draw-figures/Supplementary-Figure-12.ipynb)
(RESULTS / 'figure-data').mkdir(parents=True, exist_ok=True)
results_per_region_df.to_csv(RESULTS / 'figure-data' / 'Supplementary-Figure-12.csv', index=False)
print('Saved results_per_region_df ->', RESULTS / 'figure-data' / 'Supplementary-Figure-12.csv')

In [ ]:
# Generate the official figure from the maintained plotting notebook.
# Uses the per-fold CSV saved above; no additional model training is required.
_figure12_notebook = ROOT / '9-draw-figures' / 'Supplementary-Figure-12.ipynb'
get_ipython().safe_execfile_ipy(str(_figure12_notebook), raise_exceptions=True)
